# Traffic-signal-control experiment

This notebook compares simple baselines, MAPPO, and reward-sharing MAPPO on a network.

In [1]:
import os
import subprocess
import sys
import sysconfig
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/marl-tsc")

    if not ROOT.exists():
        subprocess.run([
            "git", "clone", "--branch", "pre-experiment2",
            "--single-branch", "https://github.com/abergh18/marl-tsc.git",
            str(ROOT)
        ], check=True)

    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "eclipse-sumo"
    ], check=True)

    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(ROOT / "requirements.txt"),
    ], check=True)

    import sumo
    os.environ["SUMO_HOME"] = sumo.SUMO_HOME
    sys.path.insert(0, os.path.join(sumo.SUMO_HOME, "tools"))

    from google.colab import userdata

    #Get access token
    token = userdata.get('GH_TOKEN')

else:
    # Run VS Code from somewhere inside the marl-tsc repository.
    ROOT = next(
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "marl_tsc").exists()
    )

# Make /src/marl_tsc importable.
sys.path.insert(0, str(ROOT / "src"))

# Relative project paths now work consistently.
os.chdir(ROOT)

print(f"Running in {'Colab' if IN_COLAB else 'VS Code'} from {ROOT}")

KeyboardInterrupt: 

In [ ]:
if not IN_COLAB:
  %reload_ext autoreload
  %autoreload 2

import matplotlib.pyplot as plt

from marl_tsc.baselines import fixed_time_actions, random_actions
from marl_tsc.mappo import train_mappo
from marl_tsc.network_types import CityNetwork, GridNetwork
from marl_tsc.simulation_generator import SimulationGenerator
from marl_tsc.training import (
    evaluate_policies,
    export_policy_replay,
    evaluation_results_table,
    plot_training_histories,
)

## 1. Configure and generate the simulation

In [ ]:
import urllib.request

if IN_COLAB:
    # Create a custom opener to disguise script as a web browser
    opener = urllib.request.build_opener()
    opener.addheaders = [
        ("User-Agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
    ]

    # Apply this rule globally for this notebook
    urllib.request.install_opener(opener)

In [ ]:
SEED = 42
EPISODE_STEPS = 600
SECONDS_PER_ACTION = 5
SIMULATION_DURATION = EPISODE_STEPS * SECONDS_PER_ACTION
TRAFFIC_SPAWN_DURATION = int(SIMULATION_DURATION * 1.20)
TOTAL_TIMESTEPS = 300_000
EVALUATION_EPISODES = 3

# The same environment settings are used for training, evaluation, and replay.
ENV_KWARGS = {
    "green_phase_count": None,
    "min_green_seconds": 10,
    "seconds_per_action": SECONDS_PER_ACTION,
    "switch_penalty": 0.1,
    "collect_global_metrics": True,
    "global_metric_interval": 10,
}

OUTPUT_DIR = ROOT / "outputs"
SIMULATION_DIR = OUTPUT_DIR / "simulation"
#network = CityNetwork(city_name="Lancaster, UK", radius=500)
network = GridNetwork(4)
generator = SimulationGenerator(
    output_dir=SIMULATION_DIR,
    network=network,
    trip_begin=0,
    trip_end=TRAFFIC_SPAWN_DURATION,
    trip_period=3,
    seed=SEED,
)

paths = generator.generate_all()
traffic_light_ids = list(paths.traffic_light_ids)
print(f"Generated SUMO config: {paths.config_file}")
print(f"Traffic-light agent count: {len(traffic_light_ids)}")

## 5. Seeded Experiments 4x4

In [ ]:
SAVE_TO_DRIVE = True

if IN_COLAB and SAVE_TO_DRIVE:
  from google.colab import drive
  drive.mount('/content/drive')
  OUTPUT_DIR = '/content/drive/MyDrive/marl_results'

In [ ]:
EXP_SWEEP = True

SEED = 42
SEED_RANGE = 7
if EXP_SWEEP:
    from marl_tsc.exp_functions import save_history, plot_variance
    import torch
    import datetime
    time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    grid= "4x4"

    #select network for seeded evaluation
    network = CityNetwork(city_name="Lancaster, UK", radius=500)
    generator = SimulationGenerator(
      output_dir=SIMULATION_DIR,
      network=network,
      trip_begin=0,
      trip_end=TRAFFIC_SPAWN_DURATION,
      trip_period=3,
      seed=SEED,
    )

    paths = generator.generate_all()
    traffic_light_ids = list(paths.traffic_light_ids)

    histories_mappo = []
    histories_rs   = []
    evaluation_histories = []

    ran_single_above = False
    if ran_single_above:
      histories_mappo.append(mappo_history)
      histories_rs.append(reward_sharing_history)
      evaluation_histories.append(policy_results)

    #Run seeded experiment sweeps
    for seed in range(SEED, SEED + SEED_RANGE):
          torch.manual_seed(seed)
          print(f"|Seed: {seed} used for sweep|")

          #Run MAPPO
          mappo_model, mappo_history, mappo_path = train_mappo(
              config_file=paths.config_file,
              traffic_light_ids=traffic_light_ids,
              output_dir=OUTPUT_DIR,
              total_timesteps=TOTAL_TIMESTEPS,
              rollout_steps=256,
              max_steps=EPISODE_STEPS,
              seed=seed,
              env_kwargs=ENV_KWARGS,
              use_peer_reward=False,
          )

          #Run reward sharing
          reward_sharing_model, reward_sharing_history, reward_sharing_path = train_mappo(
              config_file=paths.config_file,
              traffic_light_ids=traffic_light_ids,
              output_dir=OUTPUT_DIR,
              total_timesteps=TOTAL_TIMESTEPS,
              rollout_steps=256,
              max_steps=EPISODE_STEPS,
              seed=seed,
              env_kwargs=ENV_KWARGS,
              use_peer_reward=True,
          )
          #Append and save
          histories_mappo.append(mappo_history)#accumulate in memory
          histories_rs.append(reward_sharing_history)
          save_history(mappo_history, seed, f"mappo_{grid}_@{TOTAL_TIMESTEPS}steps_lr={LEARNING_RATE}_{time_stamp}", OUTPUT_DIR)#save to as backup
          save_history(reward_sharing_history, seed, f"rs__{grid}_@{TOTAL_TIMESTEPS}steps_lr={LEARNING_RATE}_{time_stamp}", OUTPUT_DIR)

          #Display
          fig, ax = plot_training_histories({
          "MAPPO": mappo_history,
          "Reward-sharing MAPPO": reward_sharing_history,
          })
          plt.show()

          #Evaluate
          policies = {
          "Random": random_actions,
          "Fixed-Time": fixed_time_actions,
          "MAPPO": mappo_model,
          "Reward-sharing MAPPO": reward_sharing_model,
          }

          policy_results = evaluate_policies(
              config_file=paths.config_file,
              traffic_light_ids=traffic_light_ids,
              policies=policies,
              episodes=EVALUATION_EPISODES,
              max_steps=EPISODE_STEPS,
              seed=SEED,
              env_kwargs=ENV_KWARGS,
          )
          evaluation_histories.append(policy_results)
          save_history(policy_results, seed, f"evaluation_{grid}_@{TOTAL_TIMESTEPS}steps_lr={LEARNING_RATE}_{time_stamp}", OUTPUT_DIR)

          display(evaluation_results_table(policy_results))
          print_gifting_summary(reward_sharing_history, traffic_light_ids)

    fig, ax = plt.subplots(figsize=(12, 5))
    plot_variance(histories_mappo, "MAPPO",              "#1565C0", ax)
    plot_variance(histories_rs,   "Reward-sharing MAPPO", "#C62828", ax)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Mean training reward")
    ax.legend()
    ax.set_title(f"Training variance across {SEED_RANGE} seeds")
    plt.tight_layout()
    plt.show()

### Results (Loaded from Drive)

In [ ]:
LOAD_FROM_DRIVE = True
if LOAD_FROM_DRIVE:
  from google.colab import drive
  drive.mount('/content/drive')
  OUTPUT_DIR = '/content/drive/MyDrive/Uni-Masters/Group Project/outputs/exp_histories'

  import json
  from pathlib import Path

  # Your OUTPUT_DIR should already point to Drive
  output_dir = Path(OUTPUT_DIR)

  # Load all histories by type
  def load_histories(pattern):
      paths = sorted(output_dir.glob(pattern))
      histories = []
      for p in paths:
          with open(p) as f:
              histories.append(json.load(f))
      print(f"Loaded {len(histories)} histories matching '{pattern}'")
      return histories

  # Training histories
  histories_mappo = load_histories("history_mappo_4x4_@300000steps*.json")
  histories_rs    = load_histories("history_rs__4x4_@300000steps*.json")
  evaluation_histories  = load_histories("history_evaluation_4x4_@300000steps*.json")

  print(f"MAPPO seeds: {len(histories_mappo)}")
  print(f"RS seeds:    {len(histories_rs)}")
  print(f"Eval seeds:  {len(evaluation_histories)}")

#### Training Graphs (Seed by Seed)

In [ ]:
if LOAD_FROM_DRIVE:
    #4x4
    for seed_idx, (mappo_hist, rs_hist) in enumerate(zip(histories_mappo, histories_rs)):
      seed = 42 + seed_idx
      print(f"\nSeed {seed}")

      # Training plots
      fig, ax = plot_training_histories({
          "MAPPO": mappo_hist,
          "Reward-sharing MAPPO": rs_hist,
      })

      plt.show()

#### Evaluation Runs


| Seed  42 | Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1211.7083 | 0.5046 | 8.3333 | 20.8957 | 244.6667 |
|  | Fixed-Time | \-986.0938 | 0.4106 | 5.0000 | 15.3450 | 89.0000 |
|  | MAPPO | \-439.6458 | 0.1831 | 4.6667 | 6.6819 | 249.6667 |
|  | Reward-sharing MAPPO | \-379.9792 | 0.1582 | 3.6667 | 3.5351 | 128.0000 |

| Seed  43 | Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1246.8542 | 0.5193 | 9.3333 | 20.0877 | 255.6667 |
|  | Fixed-Time | \-1010.1458 | 0.4206 | 6.3333 | 14.4699 | 88.3333 |
|  | MAPPO | \-550.1979 | 0.2291 | 5.0000 | 9.8022 | 276.6667 |
|  | Reward-sharing MAPPO | \-395.2500 | 0.1646 | 4.6667 | 3.4553 | 124.6667 |

| Seed  44 | Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1246.8542 | 0.5193 | 9.3333 | 20.0877 | 255.6667 |
|  | Fixed-Time | \-1010.1458 | 0.4206 | 6.3333 | 14.4699 | 88.3333 |
|  | MAPPO | \-621.6979 | 0.2589 | 5.0000 | 12.1368 | 266.0000 |
|  | Reward-sharing MAPPO | \-460.5312 | 0.1918 | 4.6667 | 6.3472 | 188.3333 |

| Seed  45 |  Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1246.8542 | 0.5193 | 9.333 | 20.0877 | 255.6667 |
|  | Fixed-Time | \-1010.1458 | 0.4206 | 6.333 | 14.4699 | 88.3333 |
|  | MAPPO | \-488.7604 | 0.2036 | 4.333 | 6.6807 | 260.6667 |
|  | Reward-sharing MAPPO | \-642.0938 | 0.2674 | 4.6667 | 11.6863 | 299.0000 |

| Seed  46 |  Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1183.3125 | 0.4928 | 9 | 20.7035 | 235.6667 |
|  | Fixed-Time | \-974.1562 | 0.4056 | 6 | 14.484 | 88.3333 |
|  | MAPPO | \-384.3958 | 0.1601 | 3.3333 | 3.26 | 115 |
|  | Reward-sharing MAPPO | \-461.3854 | 0.1921 | 4 | 6.3863 | 183.3333 |

| Seed  47 |  Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1183.3125 | 0.4928 | 9 | 20.7035 | 235.6667 |
|  | Fixed-Time | \-974.1562 | 0.4056 | 6 | 14.484 | 88.3333 |
|  | MAPPO | \-707.2292 | 0.2945 | 4.3333 | 15.2777 | 285 |
|  | Reward-sharing MAPPO | \-473.8333 | 0.1973 | 4 | 6.3599 | 164 |

| Seed  48 |  Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1183.3125 | 0.4928 | 9 | 20.7035 | 235.6667 |
|  | Fixed-Time | \-974.1562 | 0.4056 | 6 | 14.484 | 88.3333 |
|  | MAPPO | \-388.5417 | 0.1618 | 3.6667 | 3.346 | 123 |
|  | Reward-sharing MAPPO | \-637.3229 | 0.2655 | 4.3333 | 13.5747 | 254.3333 |

| Seed  49 |  Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1183.3125 | 0.4928 | 9 | 20.7035 | 235.6667 |
|  | Fixed-Time | \-974.1562 | 0.4056 | 6 | 14.484 | 88.3333 |
|  | MAPPO | \-623.2292 | 0.2595 | 4.3333 | 12.7268 | 270.3333 |
|  | Reward-sharing MAPPO | \-454.9583 | 0.1895 | 4 | 6.3847 | 283.3333 |

| Seed  50 |  Policy | Mean reward | Mean queue | Mean max queue | Mean wait | Mean max wait |
| :---: | ----- | ----- | ----- | ----- | ----- | ----- |
|  | Random | \-1183.3125 | 0.4928 | 9 | 20.7035 | 235.6667 |
|  | Fixed-Time | \-974.1562 | 0.4056 | 6 | 14.484 | 88.3333 |
|  | MAPPO | \-463.5625 | 0.193 | 3.6667 | 7.6558 | 265.3333 |
|  | Reward-sharing MAPPO | \-707.2292 | 0.2945 | 4.3333 | 15.2777 | 285 |


### Cross Seed Analysis

#### Variance Graph: Training

In [ ]:
from marl_tsc.exp_functions import plot_variance
smooth = 200
metric = "mean_training_reward"
fig, ax = plt.subplots(figsize=(12, 5))
plot_variance(histories_mappo, "MAPPO",              "#1565C0", ax, metric=metric,smooth=smooth)
plot_variance(histories_rs,   "Reward-sharing MAPPO", "#C62828", ax, metric=metric, smooth=smooth)
ax.set_xlabel("Timestep")
ax.set_ylabel(f"{metric}")
ax.legend()
ax.set_title(f"Training variance across {len(histories_rs)} seeds")
plt.tight_layout()
plt.show()

#### Statistical Significance: Training

In [ ]:
EXP_ANALYSIS = True
if EXP_ANALYSIS:
    from scipy import stats
    import numpy as np

    def training_auc(history, metric="mean_training_reward"):
        values = [h[metric] for h in history if metric in h]
        return float(np.trapezoid(values))


    def compare_training_histories(
        histories_a,
        histories_b,
        label_a="MAPPO",
        label_b="Reward-sharing MAPPO",
        metric="mean_training_reward",
        test_type="wilk",
    ):
        print(f"\nMetric: {metric}")
        print("=" * 60)

        # AUC
        aucs_a = [training_auc(h, metric) for h in histories_a]
        aucs_b = [training_auc(h, metric) for h in histories_b]
        print(f"\nAUC (total area under training curve):")
        print(f"  {label_a}: {np.mean(aucs_a):.2f} ± {np.std(aucs_a):.2f}")
        print(f"  {label_b}: {np.mean(aucs_b):.2f} ± {np.std(aucs_b):.2f}")
        if test_type == "wilk":
            stat, p = stats.wilcoxon(aucs_a, aucs_b)
        else:
            stat, p = stats.ttest_rel(aucs_a, aucs_b)

        print(f"  W={stat:.3f}, p={p:.4f}")

        if p < 0.05:
            print(f"  {label_a} and {label_b} show significantly different results")
        else:
            print(f"  {label_a} and {label_b} do not show significantly different results")

    compare_training_histories(
        histories_a=histories_mappo,
        histories_b=histories_rs,
        metric="mean_training_reward",
    )

#### Overall Performance Metrics: Evaluation

In [ ]:
if EXP_ANALYSIS:
  import numpy as np
  import pandas as pd
  metrics = ["mean_total_reward", "mean_local_queue", "mean_waiting_time",
            "mean_max_waiting_time", "mean_total_time_loss"]

  rows = []
  for policy_name in ["Random", "Fixed-Time", "MAPPO", "Reward-sharing MAPPO"]:
      row = {"Policy": policy_name}
      for metric in metrics:
          values = [e[policy_name][metric] for e in evaluation_histories]
          row[f"{metric}_mean"] = np.mean(values)
          row[f"{metric}_std"]  = np.std(values)
      rows.append(row)

  df = pd.DataFrame(rows)
  print('Overall Performance Accross 9 seeds')
  display(df)

#### Statistical Significance Tests: Evaluation

In [ ]:
if EXP_ANALYSIS:
  # Extract metric across seeds for each policy
  def perform_statisical_tests(metric, evaluation_histories, test_type):
    from scipy import stats
    mappo_grouping = [e["MAPPO"][metric] for e in evaluation_histories]
    rs_grouping   = [e["Reward-sharing MAPPO"][metric] for e in evaluation_histories]

    # Paired t-test — paired because same seed = same traffic conditions
    if test_type == "paired_t":
      _stat, p_value = stats.ttest_rel(mappo_grouping, rs_grouping)
      #print(f"t={_stat:.5f}, p={p_value:.4f}")
    elif test_type == 'wilk':
      _stat, p_value = stats.wilcoxon(mappo_grouping, rs_grouping)
      #print(f"W={_stat:.5f}, p={p_value:.4f}")
    else:
      raise ValueError(f"Unknown test type: {test_type}")

    return _stat, p_value

  metrics = ["mean_total_reward", "mean_local_queue", "mean_waiting_time",
            "mean_max_waiting_time", "mean_total_time_loss"]
  for metric in metrics:
    print(f"Metric) {metric}:")

    w, p = perform_statisical_tests(metric, evaluation_histories, "wilk")
    print(f"    Wilcoxon: t={w:.5f}, p={p:.4f}")
    if p < 0.05:
      print(f"    {metric} shows significantly different results between groups")
    else:
      print(f"    {metric} does not show significantly different results between groups ")
    print('')


In [ ]:
#print( f"Keys for mappo history: {histories_mappo[0][0].keys()}")

In [ ]:
#print(f"Keys for rs-mappo history: {histories_rs[0][0].keys()}")

In [ ]:
#print(f'''Evaluation history Keys:
#{evaluation_histories[0]['MAPPO'].keys()}''')

### Gifting Analysis

In [ ]:
from marl_tsc.evaluate_gifting import gifting_visualisation, print_gifting_summary
for seed, history in enumerate(histories_rs):
    _seed = 42 + seed
    print(f"\n{'='*50}\nSeed {_seed}\n{'='*50}")
    gifting_visualisation(
        history=history,
        agent_ids=traffic_light_ids,
        algorithm_name=f"Reward-sharing MAPPO — Seed {_seed}",
        output_dir=str(OUTPUT_DIR),
        show=True,   # don't display inline for every seed
        save=False,    # just save to Drive
    )
    print_gifting_summary(history, traffic_light_ids)

In [ ]:
!git status
#!git add .
commit = False
if commit:
  import shlex
  commit_msg = '''
  Updated to reflect gifts recieved as well as given
  '''
  !git config --global user.email user_email
  !git config --global user.name 'IsaacFayle-Waters'
  !git commit -m {shlex.quote(commit_msg)}

In [ ]:
#!git pull --rebase origin zak-pre-experiment

In [ ]:
#!git remote set-url origin https://{token}@github.com/abergh18/marl-tsc.git
#!git push -u origin zak-pre-experiment

# Learning Rate Tests (Ignore)

In [ ]:
TOTAL_TIMESTEPS = 150_000
seed = SEED
#select network for seeded evaluation
network = GridNetwork(4)
generator = SimulationGenerator(
  output_dir=SIMULATION_DIR,
  network=network,
  trip_begin=0,
  trip_end=TRAFFIC_SPAWN_DURATION,
  trip_period=3,
  seed=SEED,
)

paths = generator.generate_all()
traffic_light_ids = list(paths.traffic_light_ids)

for lr in [3e-4, 5e-4, 1e-3]:
    print(f"lr: {lr}")
    #Run MAPPO
    mappo_model, mappo_history, mappo_path = train_mappo(
        config_file=paths.config_file,
        traffic_light_ids=traffic_light_ids,
        output_dir=OUTPUT_DIR,
        total_timesteps=TOTAL_TIMESTEPS,
        rollout_steps=256,
        max_steps=EPISODE_STEPS,
        lr=lr,
        seed=seed,
        env_kwargs=ENV_KWARGS,
        use_peer_reward=False,
    )

    #Run reward sharing
    reward_sharing_model, reward_sharing_history, reward_sharing_path = train_mappo(
        config_file=paths.config_file,
        traffic_light_ids=traffic_light_ids,
        output_dir=OUTPUT_DIR,
        total_timesteps=TOTAL_TIMESTEPS,
        rollout_steps=256,
        max_steps=EPISODE_STEPS,
        lr=lr,
        seed=seed,
        env_kwargs=ENV_KWARGS,
        use_peer_reward=True,
    )

    #Display
    fig, ax = plot_training_histories({
    "MAPPO": mappo_history,
    "Reward-sharing MAPPO": reward_sharing_history,
    })
    plt.show()

    #Evaluate
    policies = {
    "Random": random_actions,
    "Fixed-Time": fixed_time_actions,
    "MAPPO": mappo_model,
    "Reward-sharing MAPPO": reward_sharing_model,
    }

    policy_results = evaluate_policies(
        config_file=paths.config_file,
        traffic_light_ids=traffic_light_ids,
        policies=policies,
        episodes=EVALUATION_EPISODES,
        max_steps=EPISODE_STEPS,
        seed=SEED,
        env_kwargs=ENV_KWARGS,
    )
